Cell 1: Setup and paths

In [6]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

RAW = Path(r"C:\Users\ASUS\ipl-business-intelligence\data\raw")
print("Folder exists:", RAW.exists())

Folder exists: True


Cell 2: Load the files

In [7]:
matches      = pd.read_csv(RAW / "matches.csv")
deliveries   = pd.read_csv(RAW / "deliveries.csv")
auction_2022 = pd.read_csv(RAW / "auction_2022.csv")
auction_2023 = pd.read_csv(RAW / "auction_2023.csv")
auction_2024 = pd.read_csv(RAW / "auction_2024.csv")

data = {
    "matches": matches, "deliveries": deliveries,
    "auction_2022": auction_2022, "auction_2023": auction_2023,
    "auction_2024": auction_2024,
}
for name, df in data.items():
    print(f"{name:<13} {df.shape}")

matches       (1095, 20)
deliveries    (260920, 17)
auction_2022  (204, 5)
auction_2023  (309, 7)
auction_2024  (675, 8)


Cell 3: Quality check on every file

In [8]:
def quick_check(df, name):
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Top missing values (%):")
    print((df.isna().mean() * 100).round(1).sort_values(ascending=False).head(5))
    print("Columns:", list(df.columns))
    print()

for name, df in data.items():
    quick_check(df, name)

--- matches ---
Shape: (1095, 20)
Duplicate rows: 0
Top missing values (%):
method             98.1
city                4.7
result_margin       1.7
player_of_match     0.5
winner              0.5
dtype: float64
Columns: ['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

--- deliveries ---
Shape: (260920, 17)
Duplicate rows: 0
Top missing values (%):
fielder             96.4
dismissal_kind      95.0
player_dismissed    95.0
extras_type         94.6
match_id             0.0
dtype: float64
Columns: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']

--- auction_2022 ---
Shape: (204, 5)
Duplicate rows: 0
Top missing values (%):
Tea

Cell 4: First rows of the three auction files

In [9]:
for name in ["auction_2022", "auction_2023", "auction_2024"]:
    print(f"--- {name} ---")
    print(list(data[name].columns))
    print(data[name].head(3).to_string())
    print()

--- auction_2022 ---
['Teams', 'Player_Name', 'Nationality', 'Type', 'Sold_Price']
                 Teams    Player_Name Nationality           Type     Sold_Price
0  Chennai Super Kings  Robin Uthappa      Indian        Batsman  ? 2,00,00,000
1  Chennai Super Kings   Dwayne Bravo    Overseas    All-Rounder  ? 4,40,00,000
2  Chennai Super Kings  Ambati Rayudu      Indian  Wicket Keeper  ? 6,75,00,000

--- auction_2023 ---
['name', 'player style', 'nationality', 'base price (in lacs)', 'final price (in lacs)', 'franchise', 'status']
            name player style   nationality  base price (in lacs)  final price (in lacs) franchise    status
0   Harshit Rana       Bowler         India                   NaN                   20.0       KKR  RETAINED
1      Ekant Sen       Batter         India                  20.0                    NaN       NaN    UNSOLD
2  Wayne Parnell   Allrounder  South Africa                  75.0                    NaN       NaN    UNSOLD

--- auction_2024 ---
['Unn

Cell 5: Teams and seasons

In [10]:
print("Teams (team1):")
for t in sorted(matches["team1"].dropna().unique()):
    print("  ", t)

print("\nSeasons:", list(matches["season"].unique()))
print("\nMatches per season:")
print(matches.groupby("season").size())

Teams (team1):
   Chennai Super Kings
   Deccan Chargers
   Delhi Capitals
   Delhi Daredevils
   Gujarat Lions
   Gujarat Titans
   Kings XI Punjab
   Kochi Tuskers Kerala
   Kolkata Knight Riders
   Lucknow Super Giants
   Mumbai Indians
   Pune Warriors
   Punjab Kings
   Rajasthan Royals
   Rising Pune Supergiant
   Rising Pune Supergiants
   Royal Challengers Bangalore
   Royal Challengers Bengaluru
   Sunrisers Hyderabad

Seasons: ['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024']

Matches per season:
season
2007/08    58
2009       57
2009/10    60
2011       73
2012       74
2013       76
2014       60
2015       59
2016       60
2017       59
2018       60
2019       60
2020/21    60
2021       60
2022       74
2023       74
2024       71
dtype: int64


Cell 6: Convert 2022 prices to crores

In [11]:
def rupee_text_to_crore(series):
    """'₹ 4,40,00,000' -> 4.4"""
    digits = series.astype(str).str.replace(r"\D", "", regex=True)
    return pd.to_numeric(digits, errors="coerce") / 1e7

auction_2022["price_cr"] = rupee_text_to_crore(auction_2022["Sold_Price"])

print("Failed conversions:", auction_2022["price_cr"].isna().sum())
print(auction_2022["price_cr"].describe())

print("\nTop 5 buys of the 2022 auction:")
auction_2022.sort_values("price_cr", ascending=False).head(5)

Failed conversions: 0
count    204.000000
mean       2.704412
std        3.323203
min        0.200000
25%        0.200000
50%        1.050000
75%        4.000000
max       15.250000
Name: price_cr, dtype: float64

Top 5 buys of the 2022 auction:


,Teams,Player_Name,Nationality,Type,Sold_Price,price_cr
100,Mumbai Indians,Ishan Kishan,Indian,Wicket Keeper,"? 15,25,00,000",15.25
3,Chennai Super Kings,Deepak Chahar,Indian,Bowler,"? 14,00,00,000",14.00
62,Kolkata Knight Riders,Shreyas Iyer,Indian,Batsman,"? 12,25,00,000",12.25
130,Punjab Kings,Liam Livingstone,Overseas,All-Rounder,"? 11,50,00,000",11.50
167,Royal Challengers Bangalore,Harshal Patel,Indian,All-Rounder,"? 10,75,00,000",10.75


Cell 7: Are retained players missing from the 2022 file?

In [12]:
name_col_2022 = [c for c in auction_2022.columns if "name" in c.lower()][0]
print("Player-name column:", name_col_2022)

check = auction_2022[auction_2022[name_col_2022].str.contains(
    "Kohli|Dhoni|Rohit|Bumrah", case=False, na=False)]
print("Rows found:", len(check))
check

Player-name column: Player_Name
Rows found: 0


,Teams,Player_Name,Nationality,Type,Sold_Price,price_cr


Cell 8: Preview the player-name mismatch

In [13]:
ball_names = set(deliveries["batter"].dropna()) | set(deliveries["bowler"].dropna())
auction_names = set(auction_2022[name_col_2022].dropna())

matched = auction_names & ball_names
unmatched = auction_names - ball_names

print("Names in ball-by-ball data:", len(ball_names))
print("Auction names:", len(auction_names))
print("Exact matches:", len(matched))
print("NOT matched:", len(unmatched))
print("\nExamples of unmatched auction names:")
print(sorted(unmatched)[:15])

Names in ball-by-ball data: 732
Auction names: 204
Exact matches: 32
NOT matched: 172

Examples of unmatched auction names:
['Abhijeet Tomar', 'Abhinav Sadarangani', 'Adam Milne', 'Aiden Markram', 'Ajinkya Rahane', 'Alex Hales', 'Alzarri Joseph', 'Aman Khan', 'Ambati Rayudu', 'Aneeshwar Gautam', 'Ankit Singh Rajpoot', 'Ansh Patel', 'Anukul Roy', 'Anunay Singh', 'Aryan Juyal']
